In [1]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
import numpy as np
from mc_engine.market.vol_surface import VolSurface
from mc_engine.calibration.heston_calibration import HestonCalibrator


In [2]:
ticker = yf.Ticker("AAPL")
prices = ticker.history("1y")
S = prices.iloc[-1].Close
r = np.log(prices.iloc[-1].Close/prices.iloc[0].Close)
q = prices.Dividends.sum()/prices.iloc[0].Close
calls = []
for date in ticker.options:
    chain = ticker.option_chain(date)
    call = chain.calls
    call["Expiry"] = date
    calls.append(call[["strike","lastPrice","Expiry","impliedVolatility"]])

calls = pd.concat(calls,axis=0,ignore_index=True)
calls["Expiry"] = pd.to_datetime(calls["Expiry"])
today = dt.datetime.now()
calls["Maturity"] = calls["Expiry"].apply(lambda x: (x - today).days/365)
calls = calls.query("Maturity > 0")
call_matrix = calls.pivot(index="Maturity",columns="strike",values="impliedVolatility").dropna(axis = 1)

In [3]:
vol_surface = VolSurface(call_matrix.columns.to_numpy(),call_matrix.index.to_numpy(),call_matrix.to_numpy())

In [5]:
HestonCalibrator(S,r,q,vol_surface).calibrate()

{'v0': np.float64(0.017535297220167643),
 'kappa': np.float64(2.002538561236375),
 'theta': np.float64(0.017538231779539257),
 'xi': np.float64(0.26498397562465065),
 'rho': np.float64(-0.6873957677274681)}